**Preprocesamiento**

In [ ]:
!pip install -q spacy openpyxl
!python -m spacy download es_core_news_sm -q
!pip install -U kaleido

print("✅ Dependencias instaladas correctamente\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 47.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Dependencias instaladas correctamente



In [ ]:
import pandas as pd
import spacy
import re
import unicodedata
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

# Cargar modelo de spaCy
nlp = spacy.load("es_core_news_sm")
print("✅ Modelo de spaCy para español cargado\n")

✅ Librerías importadas correctamente
✅ Modelo de spaCy para español cargado



In [ ]:
def limpiar_texto(texto):
    """Paso 1: Limpieza básica del texto"""
    if pd.isna(texto):
        return ""
    texto = str(texto)
    texto = texto.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

In [ ]:
def convertir_minusculas(texto):
    """Paso 2: Convertir a minúsculas"""
    if pd.isna(texto) or texto == "":
        return ""
    return texto.lower()

In [ ]:
def normalizar_unicode(texto):
    """Paso 3: Normalización UNICODE (NFD/NFC)"""
    if pd.isna(texto) or texto == "":
        return ""
    texto_nfd = unicodedata.normalize('NFD', texto)
    texto_nfc = unicodedata.normalize('NFC', texto_nfd)
    return texto_nfc

In [ ]:
def lematizar_texto(texto):
    """Paso 4: Lematización con spaCy"""
    if pd.isna(texto) or texto == "":
        return ""
    doc = nlp(texto)
    lemas = [token.lemma_ for token in doc]
    return ' '.join(lemas)

In [ ]:
def tokenizar_texto(texto):
    """Paso 5: Tokenización con spaCy"""
    if pd.isna(texto) or texto == "":
        return ""
    doc = nlp(texto)
    tokens = [token.text for token in doc if not token.is_space]
    return ', '.join(tokens)

print("✅ Funciones de procesamiento definidas\n")

✅ Funciones de procesamiento definidas



In [ ]:
ARCHIVO_ENTRADA = "/content/Corpus_humano.xlsx"

df = pd.read_excel(ARCHIVO_ENTRADA)
print(f"✅ Archivo cargado exitosamente")
print(f"📊 Total de filas: {len(df)}")
print(f"📋 Columnas encontradas: {list(df.columns)}\n")

# Normalizar nombres de columnas
df.columns = df.columns.str.strip()

# Mostrar primeras filas
print("🔍 Primeras filas del archivo original:\n")
display(df.head(2))

✅ Archivo cargado exitosamente
📊 Total de filas: 52
📋 Columnas encontradas: ['NombreArchivo', 'Resumen', 'introduccion', 'Conclusiones']

🔍 Primeras filas del archivo original:



,NombreArchivo,Resumen,introduccion,Conclusiones
0,Educación informal en emprendimiento y creatividad en escuelas innovadoras,Este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...,La educación informal ha cobrado relevancia en los últimos años como un complemento esencial a l...,La investigación evidencia que la educación informal en escuelas innovadoras desempeña un papel ...
1,Actividades inesperadas como oportunidades para la creatividad. Contextos creativos en la educac...,Este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...,"En el contexto de la educación superior argentina, caracterizado por una diversidad instituciona...","Los resultados del estudio revelan que las actividades inesperadas, lejos de ser obstáculos, pue..."


In [ ]:
# Buscar columnas de texto
columnas_procesar = []
posibles_nombres = ['Resumen', 'resumen', 'introduccion', 'Introduccion',
                    'Conclusiones', 'conclusiones']

for col in df.columns:
    if any(nombre.lower() in col.lower() for nombre in posibles_nombres):
        columnas_procesar.append(col)

print(f"✅ Columnas identificadas para procesar: {columnas_procesar}\n")

if not columnas_procesar:
    print("⚠️ No se encontraron columnas automáticamente.")
    print(f"Columnas disponibles: {list(df.columns)}")
    print("\nPor favor, especifica manualmente las columnas:")
    print("columnas_procesar = ['nombre_columna1', 'nombre_columna2']")

✅ Columnas identificadas para procesar: ['Resumen', 'introduccion', 'Conclusiones']



In [ ]:
for columna in columnas_procesar:
    col_limpia = f"{columna}_01_limpio"
    df[col_limpia] = df[columna].apply(limpiar_texto)
    print(f"✅ Procesada: {columna} → {col_limpia}")

print("\n📊 Vista previa del resultado:\n")
cols_mostrar = [columnas_procesar[0], f"{columnas_procesar[0]}_01_limpio"]
display(df[cols_mostrar].head(2))

✅ Procesada: Resumen → Resumen_01_limpio
✅ Procesada: introduccion → introduccion_01_limpio
✅ Procesada: Conclusiones → Conclusiones_01_limpio

📊 Vista previa del resultado:



,Resumen,Resumen_01_limpio
0,Este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...,Este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...
1,Este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...,Este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...


In [ ]:
for columna in columnas_procesar:
    col_minus = f"{columna}_02_minusculas"
    col_limpia = f"{columna}_01_limpio"
    df[col_minus] = df[col_limpia].apply(convertir_minusculas)
    print(f"✅ Procesada: {col_limpia} → {col_minus}")

print("\n📊 Vista previa del resultado:\n")
cols_mostrar = [f"{columnas_procesar[0]}_01_limpio",
                f"{columnas_procesar[0]}_02_minusculas"]
display(df[cols_mostrar].head(2))

✅ Procesada: Resumen_01_limpio → Resumen_02_minusculas
✅ Procesada: introduccion_01_limpio → introduccion_02_minusculas
✅ Procesada: Conclusiones_01_limpio → Conclusiones_02_minusculas

📊 Vista previa del resultado:



,Resumen_01_limpio,Resumen_02_minusculas
0,Este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...,este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...
1,Este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...,este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...


In [ ]:
for columna in columnas_procesar:
    col_unicode = f"{columna}_03_unicode"
    col_minus = f"{columna}_02_minusculas"
    df[col_unicode] = df[col_minus].apply(normalizar_unicode)
    print(f"✅ Procesada: {col_minus} → {col_unicode}")

print("\n📊 Vista previa del resultado:\n")
cols_mostrar = [f"{columnas_procesar[0]}_02_minusculas",
                f"{columnas_procesar[0]}_03_unicode"]
display(df[cols_mostrar].head(2))

✅ Procesada: Resumen_02_minusculas → Resumen_03_unicode
✅ Procesada: introduccion_02_minusculas → introduccion_03_unicode
✅ Procesada: Conclusiones_02_minusculas → Conclusiones_03_unicode

📊 Vista previa del resultado:



,Resumen_02_minusculas,Resumen_03_unicode
0,este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...,este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...
1,este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...,este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...


In [ ]:
for columna in columnas_procesar:
    col_lemas = f"{columna}_04_lematizado"
    col_unicode = f"{columna}_03_unicode"
    print(f"⏳ Procesando: {col_unicode}...")
    df[col_lemas] = df[col_unicode].apply(lematizar_texto)
    print(f"✅ Completado: {col_lemas}")

print("\n📊 Vista previa del resultado:\n")
cols_mostrar = [f"{columnas_procesar[0]}_03_unicode",
                f"{columnas_procesar[0]}_04_lematizado"]
display(df[cols_mostrar].head(2))

# Ejemplo de lematización
print("\n🔍 EJEMPLO DE LEMATIZACIÓN:")
texto_ejemplo = df[f"{columnas_procesar[0]}_03_unicode"].iloc[0][:200]
lemas_ejemplo = df[f"{columnas_procesar[0]}_04_lematizado"].iloc[0][:200]
print(f"\nOriginal: {texto_ejemplo}...")
print(f"\nLematizado: {lemas_ejemplo}...")

⏳ Procesando: Resumen_03_unicode...
✅ Completado: Resumen_04_lematizado
⏳ Procesando: introduccion_03_unicode...
✅ Completado: introduccion_04_lematizado
⏳ Procesando: Conclusiones_03_unicode...
✅ Completado: Conclusiones_04_lematizado

📊 Vista previa del resultado:



,Resumen_03_unicode,Resumen_04_lematizado
0,este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la...,este artículo analizar el papel de el educación informal en el desarrollo del emprendimiento y e...
1,este artículo explora cómo las actividades inesperadas —aquellas que surgen fuera de la planific...,este artículo explorar cómo el actividad inesperado — aquel que surgen fuera de el planificación...



🔍 EJEMPLO DE LEMATIZACIÓN:

Original: este artículo analiza el papel de la educación informal en el desarrollo del emprendimiento y la creatividad dentro de escuelas innovadoras. se parte de la premisa de que los entornos educativos no tr...

Lematizado: este artículo analizar el papel de el educación informal en el desarrollo del emprendimiento y el creatividad dentro de escuela innovadora . él partir de el premisa de que el entorno educativo no trad...


In [ ]:
for columna in columnas_procesar:
    col_tokens = f"{columna}_05_tokenizado"
    col_lemas = f"{columna}_04_lematizado"
    print(f"⏳ Procesando: {col_lemas}...")
    df[col_tokens] = df[col_lemas].apply(tokenizar_texto)
    print(f"✅ Completado: {col_tokens}")

print("\n📊 Vista previa del resultado:\n")
cols_mostrar = [f"{columnas_procesar[0]}_04_lematizado",
                f"{columnas_procesar[0]}_05_tokenizado"]
display(df[cols_mostrar].head(2))

# Mostrar estadísticas de tokens
print("\n📊 ESTADÍSTICAS DE TOKENIZACIÓN:")
for columna in columnas_procesar:
    col_tokens = f"{columna}_05_tokenizado"
    num_tokens = df[col_tokens].apply(lambda x: len(x.split(', ')) if x else 0)
    print(f"\n{columna}:")
    print(f"  - Promedio de tokens: {num_tokens.mean():.1f}")
    print(f"  - Mínimo: {num_tokens.min()}")
    print(f"  - Máximo: {num_tokens.max()}")


⏳ Procesando: Resumen_04_lematizado...
✅ Completado: Resumen_05_tokenizado
⏳ Procesando: introduccion_04_lematizado...
✅ Completado: introduccion_05_tokenizado
⏳ Procesando: Conclusiones_04_lematizado...
✅ Completado: Conclusiones_05_tokenizado

📊 Vista previa del resultado:



,Resumen_04_lematizado,Resumen_05_tokenizado
0,este artículo analizar el papel de el educación informal en el desarrollo del emprendimiento y e...,"este, artículo, analizar, el, papel, de, el, educación, informal, en, el, desarrollo, del, empre..."
1,este artículo explorar cómo el actividad inesperado — aquel que surgen fuera de el planificación...,"este, artículo, explorar, cómo, el, actividad, inesperado, —, aquel, que, surgen, fuera, de, el,..."



📊 ESTADÍSTICAS DE TOKENIZACIÓN:

Resumen:
  - Promedio de tokens: 100.8
  - Mínimo: 58
  - Máximo: 152

introduccion:
  - Promedio de tokens: 191.7
  - Mínimo: 122
  - Máximo: 269

Conclusiones:
  - Promedio de tokens: 130.7
  - Mínimo: 83
  - Máximo: 180


In [ ]:
# Mostrar información general
print("Información del DataFrame:")
print(df.info())

# Mostrar todas las columnas creadas
print("\n📋 COLUMNAS CREADAS POR CADA PASO:\n")
for columna in columnas_procesar:
    print(f"\n{columna}:")
    print(f"  01. {columna}_01_limpio")
    print(f"  02. {columna}_02_minusculas")
    print(f"  03. {columna}_03_unicode")
    print(f"  04. {columna}_04_lematizado")
    print(f"  05. {columna}_05_tokenizado")

# Mostrar DataFrame completo (con scroll horizontal)
print("\n🔍 VISTA COMPLETA DEL DATAFRAME:\n")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
display(df.head(3))

Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   NombreArchivo               52 non-null     object
 1   Resumen                     52 non-null     object
 2   introduccion                52 non-null     object
 3   Conclusiones                52 non-null     object
 4   Resumen_01_limpio           52 non-null     object
 5   introduccion_01_limpio      52 non-null     object
 6   Conclusiones_01_limpio      52 non-null     object
 7   Resumen_02_minusculas       52 non-null     object
 8   introduccion_02_minusculas  52 non-null     object
 9   Conclusiones_02_minusculas  52 non-null     object
 10  Resumen_03_unicode          52 non-null     object
 11  introduccion_03_unicode     52 non-null     object
 12  Conclusiones_03_unicode     52 non-null     object
 13  Resumen_04_lematizado    

,NombreArchivo,Resumen,introduccion,Conclusiones,Resumen_01_limpio,introduccion_01_limpio,Conclusiones_01_limpio,Resumen_02_minusculas,introduccion_02_minusculas,Conclusiones_02_minusculas,Resumen_03_unicode,introduccion_03_unicode,Conclusiones_03_unicode,Resumen_04_lematizado,introduccion_04_lematizado,Conclusiones_04_lematizado,Resumen_05_tokenizado,introduccion_05_tokenizado,Conclusiones_05_tokenizado
0,Educación informal en emprendimiento y creativ...,Este artículo analiza el papel de la educación...,La educación informal ha cobrado relevancia en...,La investigación evidencia que la educación in...,Este artículo analiza el papel de la educación...,La educación informal ha cobrado relevancia en...,La investigación evidencia que la educación in...,este artículo analiza el papel de la educación...,la educación informal ha cobrado relevancia en...,la investigación evidencia que la educación in...,este artículo analiza el papel de la educación...,la educación informal ha cobrado relevancia en...,la investigación evidencia que la educación in...,este artículo analizar el papel de el educació...,el educación informal haber cobrar relevancia ...,el investigación evidenciar que el educación i...,"este, artículo, analizar, el, papel, de, el, e...","el, educación, informal, haber, cobrar, releva...","el, investigación, evidenciar, que, el, educac..."
1,Actividades inesperadas como oportunidades par...,Este artículo explora cómo las actividades ine...,En el contexto de la educación superior argent...,Los resultados del estudio revelan que las act...,Este artículo explora cómo las actividades ine...,En el contexto de la educación superior argent...,Los resultados del estudio revelan que las act...,este artículo explora cómo las actividades ine...,en el contexto de la educación superior argent...,los resultados del estudio revelan que las act...,este artículo explora cómo las actividades ine...,en el contexto de la educación superior argent...,los resultados del estudio revelan que las act...,este artículo explorar cómo el actividad inesp...,en el contexto de el educación superior argent...,el resultado del estudio revelar que el activi...,"este, artículo, explorar, cómo, el, actividad,...","en, el, contexto, de, el, educación, superior,...","el, resultado, del, estudio, revelar, que, el,..."
2,Aprendizaje Basado en un Proyecto Docente: Apr...,Este artículo analiza el impacto del Aprendiza...,La formación de docentes en la era digital exi...,El estudio concluye que el ABP representa una ...,Este artículo analiza el impacto del Aprendiza...,La formación de docentes en la era digital exi...,El estudio concluye que el ABP representa una ...,este artículo analiza el impacto del aprendiza...,la formación de docentes en la era digital exi...,el estudio concluye que el abp representa una ...,este artículo analiza el impacto del aprendiza...,la formación de docentes en la era digital exi...,el estudio concluye que el abp representa una ...,este artículo analizar el impacto del aprendiz...,el formación de docente en el ser digital exig...,el estudio concluir que el abp representar uno...,"este, artículo, analizar, el, impacto, del, ap...","el, formación, de, docente, en, el, ser, digit...","el, estudio, concluir, que, el, abp, represent..."


In [ ]:
df

,NombreArchivo,Resumen,introduccion,Conclusiones,Resumen_01_limpio,introduccion_01_limpio,Conclusiones_01_limpio,Resumen_02_minusculas,introduccion_02_minusculas,Conclusiones_02_minusculas,Resumen_03_unicode,introduccion_03_unicode,Conclusiones_03_unicode,Resumen_04_lematizado,introduccion_04_lematizado,Conclusiones_04_lematizado,Resumen_05_tokenizado,introduccion_05_tokenizado,Conclusiones_05_tokenizado
0,Educación informal en emprendimiento y creativ...,Este artículo analiza el papel de la educación...,La educación informal ha cobrado relevancia en...,La investigación evidencia que la educación in...,Este artículo analiza el papel de la educación...,La educación informal ha cobrado relevancia en...,La investigación evidencia que la educación in...,este artículo analiza el papel de la educación...,la educación informal ha cobrado relevancia en...,la investigación evidencia que la educación in...,este artículo analiza el papel de la educación...,la educación informal ha cobrado relevancia en...,la investigación evidencia que la educación in...,este artículo analizar el papel de el educació...,el educación informal haber cobrar relevancia ...,el investigación evidenciar que el educación i...,"este, artículo, analizar, el, papel, de, el, e...","el, educación, informal, haber, cobrar, releva...","el, investigación, evidenciar, que, el, educac..."
1,Actividades inesperadas como oportunidades par...,Este artículo explora cómo las actividades ine...,En el contexto de la educación superior argent...,Los resultados del estudio revelan que las act...,Este artículo explora cómo las actividades ine...,En el contexto de la educación superior argent...,Los resultados del estudio revelan que las act...,este artículo explora cómo las actividades ine...,en el contexto de la educación superior argent...,los resultados del estudio revelan que las act...,este artículo explora cómo las actividades ine...,en el contexto de la educación superior argent...,los resultados del estudio revelan que las act...,este artículo explorar cómo el actividad inesp...,en el contexto de el educación superior argent...,el resultado del estudio revelar que el activi...,"este, artículo, explorar, cómo, el, actividad,...","en, el, contexto, de, el, educación, superior,...","el, resultado, del, estudio, revelar, que, el,..."
2,Aprendizaje Basado en un Proyecto Docente: Apr...,Este artículo analiza el impacto del Aprendiza...,La formación de docentes en la era digital exi...,El estudio concluye que el ABP representa una ...,Este artículo analiza el impacto del Aprendiza...,La formación de docentes en la era digital exi...,El estudio concluye que el ABP representa una ...,este artículo analiza el impacto del aprendiza...,la formación de docentes en la era digital exi...,el estudio concluye que el abp representa una ...,este artículo analiza el impacto del aprendiza...,la formación de docentes en la era digital exi...,el estudio concluye que el abp representa una ...,este artículo analizar el impacto del aprendiz...,el formación de docente en el ser digital exig...,el estudio concluir que el abp representar uno...,"este, artículo, analizar, el, impacto, del, ap...","el, formación, de, docente, en, el, ser, digit...","el, estudio, concluir, que, el, abp, represent..."
3,Aprendizaje significativo apoyado en la creati...,Este artículo examina cómo la creatividad y la...,"El aprendizaje significativo, conceptualizado ...",Los hallazgos del estudio indican que la creat...,Este artículo examina cómo la creatividad y la...,"El aprendizaje significativo, conceptualizado ...",Los hallazgos del estudio indican que la creat...,este artículo examina cómo la creatividad y la...,"el aprendizaje significativo, conceptualizado ...",los hallazgos del estudio indican que la creat...,este artículo examina cómo la creatividad y la...,"el aprendizaje significativo, conceptualizado ...",los hallazgos del estudio indican que la creat...,este artículo examina cómo el creatividad y el...,"el apre

In [ ]:
ARCHIVO_SALIDA = "/content/Corpus_humano_procesado.xlsx"

print(f"Guardando en: {ARCHIVO_SALIDA}\n")

# Guardar con formato
with pd.ExcelWriter(ARCHIVO_SALIDA, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Datos Procesados')

    workbook = writer.book
    worksheet = writer.sheets['Datos Procesados']

    # Ajustar ancho de columnas
    for column in worksheet.columns:
        max_length = 0
        column = [cell for cell in column]
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(cell.value)
            except:
                pass
        adjusted_width = min(max_length + 2, 60)
        worksheet.column_dimensions[column[0].column_letter].width = adjusted_width

    # Aplicar colores y formato a encabezados
    for idx, col in enumerate(df.columns, start=1):
        cell = worksheet.cell(row=1, column=idx)
        cell.font = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

        if '_01_limpio' in col:
            cell.fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
        elif '_02_minusculas' in col:
            cell.fill = PatternFill(start_color="70AD47", end_color="70AD47", fill_type="solid")
        elif '_03_unicode' in col:
            cell.fill = PatternFill(start_color="FFC000", end_color="FFC000", fill_type="solid")
        elif '_04_lematizado' in col:
            cell.fill = PatternFill(start_color="ED7D31", end_color="ED7D31", fill_type="solid")
        elif '_05_tokenizado' in col:
            cell.fill = PatternFill(start_color="5B9BD5", end_color="5B9BD5", fill_type="solid")
        else:
            cell.fill = PatternFill(start_color="7030A0", end_color="7030A0", fill_type="solid")

print("✅ Archivo guardado exitosamente")
print(f"📁 Ubicación: {ARCHIVO_SALIDA}\n")

# Descargar archivo (opcional en Colab)
from google.colab import files
print("⬇️ ¿Deseas descargar el archivo? Descomenta la siguiente línea:\n")
print("# files.download(ARCHIVO_SALIDA)")

print("\n" + "="*70)
print("✅ PROCESAMIENTO COMPLETADO EXITOSAMENTE")
print("="*70)
print("\n🎉 Tu corpus ha sido procesado paso a paso con éxito!")
print("📊 Cada paso está en columnas paralelas para análisis detallado")
print("🔬 Listo para tu estudio de caracterización del discurso\n")

Guardando en: /content/Corpus_humano_procesado.xlsx

✅ Archivo guardado exitosamente
📁 Ubicación: /content/Corpus_humano_procesado.xlsx

⬇️ ¿Deseas descargar el archivo? Descomenta la siguiente línea:

# files.download(ARCHIVO_SALIDA)

✅ PROCESAMIENTO COMPLETADO EXITOSAMENTE

🎉 Tu corpus ha sido procesado paso a paso con éxito!
📊 Cada paso está en columnas paralelas para análisis detallado
🔬 Listo para tu estudio de caracterización del discurso



**Análisis semántico**

In [ ]:
import sys
import subprocess

def instalar_paquete(paquete):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

# Lista de paquetes necesarios
paquetes = [
    "scikit-learn>=1.0.0",
    "gensim>=4.0.0",
    "wordcloud",
    "matplotlib>=3.3.0",
    "seaborn>=0.11.0",
    "sentence-transformers",
    "openpyxl",
    "spacy"
]

for paquete in paquetes:
    try:
        instalar_paquete(paquete)
    except:
        print(f"⚠️  Error instalando {paquete}, continuando...")

print("✅ Dependencias instaladas\n")

✅ Dependencias instaladas



In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Backend no interactivo para evitar problemas
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from wordcloud import WordCloud

# Para topic modeling
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# Gensim
from gensim import corpora
from gensim.models import LdaModel

# Sentence transformers
from sentence_transformers import SentenceTransformer

# Configuración de visualización
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', None)

print("✅ Librerías importadas correctamente\n")

✅ Librerías importadas correctamente



In [ ]:
# Cargar stopwords de spaCy
try:
    import spacy
    from spacy.lang.es.stop_words import STOP_WORDS
    print("✅ spaCy cargado correctamente")
except:
    print("⚠️  Instalando modelo de spaCy...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "es_core_news_sm"])
    import spacy
    from spacy.lang.es.stop_words import STOP_WORDS
    print("✅ spaCy instalado y cargado")

nlp = spacy.load("es_core_news_sm")

# Cargar datos
ARCHIVO_PROCESADO = "/content/Corpus_humano_procesado.xlsx"

try:
    df = pd.read_excel(ARCHIVO_PROCESADO, engine='openpyxl')
    print(f"✅ Archivo cargado: {len(df)} documentos")
except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo {ARCHIVO_PROCESADO}")
    print("   Asegúrate de ejecutar primero el script de preprocesamiento")
    raise

# Identificar columnas lematizadas
columnas_lematizadas = [col for col in df.columns if '_04_lematizado' in col]
print(f"🎯 Columnas para análisis: {len(columnas_lematizadas)}\n")

✅ spaCy cargado correctamente
✅ Archivo cargado: 52 documentos
🎯 Columnas para análisis: 3



In [ ]:
# Combinar todas las columnas lematizadas en un solo texto por documento
df['texto_completo_lematizado'] = ''

for col in columnas_lematizadas:
    df['texto_completo_lematizado'] += ' ' + df[col].fillna('')

df['texto_completo_lematizado'] = df['texto_completo_lematizado'].str.strip()

print(f"✅ Texto completo creado para {len(df)} documentos")
print(f"📊 Longitud promedio: {df['texto_completo_lematizado'].str.len().mean():.0f} caracteres\n")

# Mostrar ejemplo
print("🔍 Ejemplo del texto completo (primeros 300 caracteres):\n")
print(df['texto_completo_lematizado'].iloc[0][:300] + "...\n")


✅ Texto completo creado para 52 documentos
📊 Longitud promedio: 2615 caracteres

🔍 Ejemplo del texto completo (primeros 300 caracteres):

este artículo analizar el papel de el educación informal en el desarrollo del emprendimiento y el creatividad dentro de escuela innovadora . él partir de el premisa de que el entorno educativo no tradicional , como taller extracurricular , espacio maker , club de innovación y proyecto colaborativo ,...



In [ ]:
# Función para contar palabras sin stopwords
def contar_palabras_sin_stopwords(texto):
    """Cuenta palabras excluyendo stopwords"""
    if pd.isna(texto) or texto == "":
        return []

    palabras = texto.lower().split()
    # Filtrar stopwords y palabras muy cortas
    palabras_filtradas = [p for p in palabras
                          if p not in STOP_WORDS
                          and len(p) > 2
                          and p.isalpha()]
    return palabras_filtradas

# Aplicar a todo el corpus
todas_palabras_sin_stop = []
for texto in df['texto_completo_lematizado']:
    todas_palabras_sin_stop.extend(contar_palabras_sin_stopwords(texto))

# Contar frecuencias
frecuencias_sin_stop = Counter(todas_palabras_sin_stop)

print(f" Total de palabras (sin stopwords): {len(todas_palabras_sin_stop):,}")
print(f" Vocabulario único (sin stopwords): {len(frecuencias_sin_stop):,} palabras\n")

# Top 20 palabras sin stopwords
top20_sin_stop = frecuencias_sin_stop.most_common(20)

print(" TOP 20 PALABRAS MÁS FRECUENTES (SIN STOPWORDS):\n")
df_top20_sin_stop = pd.DataFrame(top20_sin_stop, columns=['Palabra', 'Frecuencia'])
df_top20_sin_stop['Ranking'] = range(1, 21)
df_top20_sin_stop = df_top20_sin_stop[['Ranking', 'Palabra', 'Frecuencia']]
display(df_top20_sin_stop)

# Guardar para análisis posterior
df['palabras_sin_stopwords'] = df['texto_completo_lematizado'].apply(
    lambda x: ' '.join(contar_palabras_sin_stopwords(x))
)


 Total de palabras (sin stopwords): 10,272
 Vocabulario único (sin stopwords): 1,425 palabras

 TOP 20 PALABRAS MÁS FRECUENTES (SIN STOPWORDS):



,Ranking,Palabra,Frecuencia
0,1,creatividad,392
1,2,creativo,148
2,3,estudio,135
3,4,educativo,129
4,5,desarrollo,115
5,6,analizar,102
6,7,docente,101
7,8,educación,99
8,9,promover,89
9,10,estudiante,88


In [ ]:
# Función para contar todas las palabras
def contar_todas_palabras(texto):
    """Cuenta todas las palabras incluyendo stopwords"""
    if pd.isna(texto) or texto == "":
        return []

    palabras = texto.lower().split()
    # Filtrar solo palabras alfabéticas de más de 2 caracteres
    palabras_filtradas = [p for p in palabras if len(p) > 2 and p.isalpha()]
    return palabras_filtradas

# Aplicar a todo el corpus
todas_palabras_con_stop = []
for texto in df['texto_completo_lematizado']:
    todas_palabras_con_stop.extend(contar_todas_palabras(texto))

# Contar frecuencias
frecuencias_con_stop = Counter(todas_palabras_con_stop)

print(f" Total de palabras (con stopwords): {len(todas_palabras_con_stop):,}")
print(f" Vocabulario único (con stopwords): {len(frecuencias_con_stop):,} palabras\n")

# Top 20 palabras con stopwords
top20_con_stop = frecuencias_con_stop.most_common(20)

print(" TOP 20 PALABRAS MÁS FRECUENTES (CON STOPWORDS):\n")
df_top20_con_stop = pd.DataFrame(top20_con_stop, columns=['Palabra', 'Frecuencia'])
df_top20_con_stop['Ranking'] = range(1, 21)
df_top20_con_stop = df_top20_con_stop[['Ranking', 'Palabra', 'Frecuencia']]
display(df_top20_con_stop)


 Total de palabras (con stopwords): 13,159
 Vocabulario único (con stopwords): 1,517 palabras

 TOP 20 PALABRAS MÁS FRECUENTES (CON STOPWORDS):



,Ranking,Palabra,Frecuencia
0,1,que,485
1,2,creatividad,392
2,3,uno,321
3,4,como,306
4,5,del,186
5,6,este,172
6,7,para,171
7,8,creativo,148
8,9,estudio,135
9,10,educativo,129


In [ ]:
import matplotlib.pyplot as plt

# Crear figura y ejes
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Sin stopwords
palabras_sin = [p[0] for p in top20_sin_stop]
frecuencias_sin = [p[1] for p in top20_sin_stop]

axes[0].barh(palabras_sin[::-1], frecuencias_sin[::-1], color='steelblue')
axes[0].set_xlabel('Frecuencia', fontsize=12)
axes[0].set_title('Top 20 Palabras (SIN Stopwords)', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Gráfico 2: Con stopwords
palabras_con = [p[0] for p in top20_con_stop]
frecuencias_con = [p[1] for p in top20_con_stop]

axes[1].barh(palabras_con[::-1], frecuencias_con[::-1], color='coral')
axes[1].set_xlabel('Frecuencia', fontsize=12)
axes[1].set_title('Top 20 Palabras (CON Stopwords)', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()

# Mostrar primero el gráfico
plt.show()

# Luego guardarlo
fig.savefig('/content/top20_palabras_comparacion.png', dpi=300, bbox_inches='tight')

print("✅ Gráfico mostrado y guardado como: top20_palabras_comparacion.png")


✅ Gráfico mostrado y guardado como: top20_palabras_comparacion.png


In [ ]:
# Crear nube de palabras (sin stopwords)
texto_completo = ' '.join(todas_palabras_sin_stop)

wordcloud = WordCloud(width=1200, height=600,
                      background_color='white',
                      colormap='viridis',
                      max_words=100,
                      relative_scaling=0.5).generate(texto_completo)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Nube de Palabras - Corpus Completo (Sin Stopwords)',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/content/nube_palabras.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Nube de palabras guardada: nube_palabras.png\n")

✅ Nube de palabras guardada: nube_palabras.png



In [ ]:
def extraer_oraciones(texto):
    """Extrae oraciones del texto (delimitadas por puntos)"""
    if pd.isna(texto) or texto == "":
        return []

    # Dividir por puntos, signos de interrogación y exclamación
    oraciones = re.split(r'[.!?]+', texto)

    # Limpiar y filtrar oraciones vacías o muy cortas
    oraciones_limpias = []
    for oracion in oraciones:
        oracion = oracion.strip()
        # Filtrar oraciones con al menos 5 palabras
        if len(oracion.split()) >= 5:
            oraciones_limpias.append(oracion)

    return oraciones_limpias

# Extraer todas las oraciones del corpus
todas_oraciones = []
for texto in df['texto_completo_lematizado']:
    todas_oraciones.extend(extraer_oraciones(texto))

print(f"✅ Total de oraciones extraídas: {len(todas_oraciones):,}")

# Contar frecuencias de oraciones
frecuencias_oraciones = Counter(todas_oraciones)

print(f"✅ Oraciones únicas: {len(frecuencias_oraciones):,}\n")

# Top 20 oraciones más frecuentes
top20_oraciones = frecuencias_oraciones.most_common(20)

print("🏆 TOP 20 ORACIONES MÁS FRECUENTES:\n")
df_top20_oraciones = pd.DataFrame(top20_oraciones,
                                  columns=['Oración', 'Frecuencia'])
df_top20_oraciones['Ranking'] = range(1, 21)
df_top20_oraciones = df_top20_oraciones[['Ranking', 'Oración', 'Frecuencia']]

# Truncar oraciones largas para mejor visualización
df_top20_oraciones['Oración_corta'] = df_top20_oraciones['Oración'].apply(
    lambda x: x[:80] + '...' if len(x) > 80 else x
)

display(df_top20_oraciones[['Ranking', 'Oración_corta', 'Frecuencia']])

# Guardar oraciones completas
print("\n📄 Oraciones completas del Top 5:\n")
for i, (oracion, freq) in enumerate(top20_oraciones[:5], 1):
    print(f"{i}. [{freq} veces] {oracion}\n")

✅ Total de oraciones extraídas: 789
✅ Oraciones únicas: 789

🏆 TOP 20 ORACIONES MÁS FRECUENTES:



,Ranking,Oración_corta,Frecuencia
0,1,este artículo analizar el papel de el educación informal en el desarrollo del em...,1
1,2,"él partir de el premisa de que el entorno educativo no tradicional , como taller...",1
2,3,a través de uno revisión teórico y estudio de caso en centro educativo de enfoqu...,1
3,4,el trabajo concluir que el educación informal no solo complementar el formación ...,1
4,5,el educación informal haber cobrar relevancia en el último año como uno compleme...,1
5,6,"en escuela innovadora , caracterizado por su apertura a metodología activo , fle...",1
6,7,el objetivo de este artículo ser analizar cómo este práctica informal contribuir...,1
7,8,el justificación radicar en el necesidad de formar individuo capaz de adaptar él...,1
8,9,el marco teórico él fundamentar en el teoría del aprendizaje experiencial ( kolb...,1
9,10,"asimismo , él considerar estudio reciente sobre innovación educativo y emprendim...",1



📄 Oraciones completas del Top 5:

1. [1 veces] este artículo analizar el papel de el educación informal en el desarrollo del emprendimiento y el creatividad dentro de escuela innovadora

2. [1 veces] él partir de el premisa de que el entorno educativo no tradicional , como taller extracurricular , espacio maker , club de innovación y proyecto colaborativo , fomentar habilidad emprendedor y creativa que no siempre él cultivar en el currículo formal

3. [1 veces] a través de uno revisión teórico y estudio de caso en centro educativo de enfoque alternativo , él identificir práctica pedagógico que promover el autonomía , el resolución de problema , el pensamiento divergente y el iniciativa personal

4. [1 veces] el trabajo concluir que el educación informal no solo complementar el formación académico , sino que poder ser uno catalizador clave para el desarrollo de competencia emprendedor y creativa en contexto escolar que apostar por el innovación

5. [1 veces] el educación informal haber

In [ ]:
# Cargar modelo ligero multilingüe (incluye español)
# Este modelo es relativamente pequeño (~120MB)
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("✅ Modelo de embeddings cargado correctamente\n")

# Generar embeddings para las palabras más frecuentes
print("🔄 Generando embeddings para Top 50 palabras...\n")

top50_palabras = [palabra for palabra, _ in frecuencias_sin_stop.most_common(50)]
embeddings_palabras = modelo_embeddings.encode(top50_palabras)

print(f"✅ Embeddings generados: {embeddings_palabras.shape}")
print(f"   - {embeddings_palabras.shape[0]} palabras")
print(f"   - {embeddings_palabras.shape[1]} dimensiones por embedding\n")

# Calcular similitudes entre palabras
from sklearn.metrics.pairwise import cosine_similarity

print("🔍 Calculando similitudes entre palabras...\n")

similitudes = cosine_similarity(embeddings_palabras)

# Función para encontrar palabras similares
def encontrar_similares(palabra, top_n=5):
    """Encuentra las palabras más similares a una palabra dada"""
    if palabra not in top50_palabras:
        return f"'{palabra}' no está en el Top 50"

    idx = top50_palabras.index(palabra)
    similitudes_palabra = similitudes[idx]

    # Ordenar por similitud (excluyendo la palabra misma)
    indices_similares = np.argsort(similitudes_palabra)[::-1][1:top_n+1]

    resultados = []
    for i in indices_similares:
        resultados.append((top50_palabras[i], similitudes_palabra[i]))

    return resultados

# Ejemplo: palabras similares a las 3 más frecuentes
print("🎯 PALABRAS SIMILARES (basadas en embeddings):\n")

for i in range(min(3, len(top50_palabras))):
    palabra = top50_palabras[i]
    similares = encontrar_similares(palabra, top_n=5)

    print(f"'{palabra}' es similar a:")
    for sim_palabra, sim_score in similares:
        print(f"  • {sim_palabra} (similitud: {sim_score:.3f})")
    print()

# Guardar embeddings en el DataFrame (solo para palabras top)
print("💾 Guardando información de embeddings...\n")

# Crear diccionario de embeddings
dict_embeddings = {palabra: emb for palabra, emb in zip(top50_palabras, embeddings_palabras)}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modelo de embeddings cargado correctamente

🔄 Generando embeddings para Top 50 palabras...

✅ Embeddings generados: (50, 384)
   - 50 palabras
   - 384 dimensiones por embedding

🔍 Calculando similitudes entre palabras...

🎯 PALABRAS SIMILARES (basadas en embeddings):

'creatividad' es similar a:
  • creativo (similitud: 0.931)
  • generar (similitud: 0.670)
  • proyecto (similitud: 0.658)
  • explorar (similitud: 0.652)
  • estudio (similitud: 0.638)

'creativo' es similar a:
  • creatividad (similitud: 0.931)
  • generar (similitud: 0.790)
  • proyecto (similitud: 0.766)
  • fomentar (similitud: 0.759)
  • estudio (similitud: 0.722)

'estudio' es similar a:
  • analizar (similitud: 0.893)
  • generar (similitud: 0.850)
  • práctica (similitud: 0.849)
  • revisión (similitud: 0.814)
  • explorar (similitud: 0.804)

💾 Guardando información de embeddings...



In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()

palabras_base = top50_palabras[:3]

for palabra in palabras_base:
    similares = encontrar_similares(palabra, top_n=5)
    for sim_palabra, sim_score in similares:
        G.add_edge(palabra, sim_palabra, weight=sim_score)

pos = nx.spring_layout(G, k=0.8, seed=42)

plt.figure(figsize=(10, 8))

nx.draw_networkx_nodes(G, pos, node_size=800)

edges = G.edges(data=True)
weights = [d['weight'] * 3 for (_, _, d) in edges]

nx.draw_networkx_edges(G, pos, width=weights)
nx.draw_networkx_labels(G, pos, font_size=10)

plt.title("Relaciones de similitud")

# 🔥 GUARDAR IMAGEN
plt.savefig("grafo_similitud.png", dpi=300, bbox_inches='tight')

plt.close()  # 👈 importante para que no intente mostrarla

## Grafo dirigido

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()

palabras_base = top50_palabras[:3]

for palabra in palabras_base:
    similares = encontrar_similares(palabra, top_n=5)
    for sim_palabra, sim_score in similares:
        G.add_edge(palabra, sim_palabra, weight=sim_score)

pos = nx.spring_layout(G, k=0.8, seed=42)

plt.figure(figsize=(10, 8))

nx.draw(G, pos, with_labels=True, node_size=800, arrows=True)

plt.title("Relaciones dirigidas de similitud")
plt.savefig("grafo_dirigido.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 🔹 1. Crear grafo dirigido
G = nx.DiGraph()

palabras_base = top50_palabras[:3]

for palabra in palabras_base:
    similares = encontrar_similares(palabra, top_n=5)  # puedes bajar a 3 si se ve cargado
    for sim_palabra, sim_score in similares:
        G.add_edge(palabra, sim_palabra, weight=sim_score)

# 🔹 2. Calcular centralidad (grado)
centralidad = nx.degree_centrality(G)

# 🔹 3. Tamaños de nodos según centralidad
node_sizes = [3000 * centralidad[nodo] + 300 for nodo in G.nodes()]

# 🔹 4. Colores según importancia
node_colors = []
for nodo in G.nodes():
    if centralidad[nodo] > 0.5:
        node_colors.append("#d62728")  # rojo
    elif centralidad[nodo] > 0.3:
        node_colors.append("#ff7f0e")  # naranja
    else:
        node_colors.append("#1f77b4")  # azul

# 🔹 5. Layout (distribución)
pos = nx.spring_layout(G, k=0.8, seed=42)

# 🔹 6. Graficar
plt.figure(figsize=(12, 10))

# EDGES (primero para que no queden ocultos)
nx.draw_networkx_edges(
    G, pos,
    arrows=True,
    arrowstyle='-|>',
    arrowsize=20,
    width=1.5,
    alpha=1,
    min_source_margin=10,
    min_target_margin=10
)

# NODOS (con transparencia y borde)
nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors='black',
    linewidths=1,
    alpha=0.60
)

# ETIQUETAS
nx.draw_networkx_labels(
    G, pos,
    font_size=10,
    font_weight='bold'
)

# 🔹 7. Título y formato
plt.title("Grafo dirigido de similitud semántica con centralidad", fontsize=14)
plt.axis("off")

# 🔹 8. Guardar imagen
plt.savefig("grafo_centralidad_mejorado.png", dpi=300, bbox_inches='tight')
plt.close()

# 🔹 9. Mostrar nodos más importantes
top_centrales = sorted(centralidad.items(), key=lambda x: x[1], reverse=True)

print("🔝 Nodos más centrales:")
for nodo, valor in top_centrales[:5]:
    print(f"{nodo}: {valor:.3f}")

🔝 Nodos más centrales:
estudio: 0.778
creatividad: 0.667
creativo: 0.667
generar: 0.333
proyecto: 0.222


In [ ]:
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import plotly.express as px

# 🔹 1. Subconjunto de palabras (base + similares)
palabras_base = top50_palabras[:3]
sub_palabras = set(palabras_base)

for palabra in palabras_base:
    similares = encontrar_similares(palabra, top_n=5)
    for sim_palabra, _ in similares:
        sub_palabras.add(sim_palabra)

sub_palabras = list(sub_palabras)

# 🔹 2. Obtener embeddings
indices = [top50_palabras.index(p) for p in sub_palabras]
emb_subset = embeddings_palabras[indices]

# 🔹 3. Reducir dimensiones (t-SNE)
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
emb_2d = tsne.fit_transform(emb_subset)

# 🔹 4. Clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(emb_subset)

# 🔹 5. Tamaño (usa frecuencia real si la tienes)
sizes = []
for palabra in sub_palabras:
    if palabra in frecuencias_sin_stop:
        sizes.append(frecuencias_sin_stop[palabra])
    else:
        sizes.append(1)

sizes = np.array(sizes)
sizes = (sizes - sizes.min() + 1) * 10  # normalizar tamaños

# 🔹 6. DataFrame
df = pd.DataFrame({
    "x": emb_2d[:, 0],
    "y": emb_2d[:, 1],
    "palabra": sub_palabras,
    "cluster": clusters,
    "size": sizes
})

# =========================================================
# 🔥 PARTE 1: INTERACTIVO (Plotly)
# =========================================================

fig = px.scatter(
    df, x="x", y="y",
    text="palabra",
    size="size",
    color="cluster",
    title="Clusters semánticos (Interactive Bubble)"
)

fig.update_traces(textposition='top center')

# Guardar HTML (estable)
fig.write_html("bubble_clusters.html")

fig.show()

# =========================================================
# 🔥 PARTE 2: PNG PARA TESIS (Matplotlib)
# =========================================================

plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    df["x"], df["y"],
    s=df["size"] * 10,
    c=df["cluster"]
)

# etiquetas
for i, txt in enumerate(df["palabra"]):
    plt.annotate(txt, (df["x"][i], df["y"][i]), fontsize=9)

plt.title("Clusters semánticos (Bubble - PNG)")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")

plt.savefig("bubble_clusters.png", dpi=300, bbox_inches='tight')
plt.close()

print("✅ Archivos generados:")
print("- bubble_clusters.html (interactivo)")
print("- bubble_clusters.png (para tesis)")

✅ Archivos generados:
- bubble_clusters.html (interactivo)
- bubble_clusters.png (para tesis)


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# 🔹 Crear grafo dirigido
G = nx.DiGraph()

palabras_base = top50_palabras[:3]

for palabra in palabras_base:
    similares = encontrar_similares(palabra, top_n=5)  # 🔥 menos conexiones
    for sim_palabra, sim_score in similares:
        G.add_edge(palabra, sim_palabra, weight=sim_score)

# 🔹 Posiciones desde t-SNE
pos = {palabra: (df[df["palabra"] == palabra]["x"].values[0],
                 df[df["palabra"] == palabra]["y"].values[0])
       for palabra in df["palabra"]}

plt.figure(figsize=(10, 8))
node_size=df["size"] * 10

#Mapear tamaños correctamente
size_map = {
    row["palabra"]: row["size"]
    for _, row in df.iterrows()
}

node_sizes = [size_map[node] * 10 for node in G.nodes()]

color_map = {
    row["palabra"]: row["cluster"]
    for _, row in df.iterrows()
}

node_colors = [color_map[node] for node in G.nodes()]

# 🔹 Nodos (bubble corregido)
nx.draw_networkx_nodes(
    G, pos,
    node_size= node_sizes,
    node_color= node_colors,
    alpha=0.9
)
# 🔹 Flechas (relaciones)
edges = G.edges(data=True)
weights = [d['weight'] * 2 for (_, _, d) in edges]

nx.draw_networkx_edges(
    G, pos,
    arrowstyle='->',
    arrowsize=15,
    width=weights,
    alpha=0.6
)

# 🔹 Labels
nx.draw_networkx_labels(G, pos, font_size=9)

plt.title("Relaciones semánticas (Clusters + Grafo dirigido)")
plt.axis("off")

plt.savefig("grafo_bubble_hibrido.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    df["x"], df["y"],
    s=df["size"] * 10,
    c=df["cluster"]
)

for i, txt in enumerate(df["palabra"]):
    plt.annotate(txt, (df["x"][i], df["y"][i]), fontsize=9)

plt.title("Distribución semántica de palabras basada en embeddings")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")

plt.savefig("grafica_principal_tesis.png", dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
import plotly.express as px
import pandas as pd
from sklearn.manifold import TSNE

# Subconjunto de palabras (igual que antes)
palabras_base = top50_palabras[:3]
sub_palabras = set(palabras_base)

for palabra in palabras_base:
    similares = encontrar_similares(palabra, top_n=5)
    for sim_palabra, _ in similares:
        sub_palabras.add(sim_palabra)

sub_palabras = list(sub_palabras)

# índices
indices = [top50_palabras.index(p) for p in sub_palabras]
emb_subset = embeddings_palabras[indices]

# Reducir dimensiones
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
emb_2d = tsne.fit_transform(emb_subset)

# Tamaño del globo (puedes usar frecuencia si la tienes)
sizes = [10 + 5*i for i in range(len(sub_palabras))]

df = pd.DataFrame({
    "x": emb_2d[:, 0],
    "y": emb_2d[:, 1],
    "palabra": sub_palabras,
    "size": sizes
})

fig = px.scatter(df, x="x", y="y", text="palabra", size="size",
                 title="Nube de palabras (Bubble chart)")

fig.update_traces(textposition='top center')
fig.write_image("bubble_chart.png")  # guardar
fig.show()

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Reducir dimensionalidad con PCA
print("⏳ Reduciendo dimensionalidad con PCA...\n")
pca = PCA(n_components=2)
embeddings_2d_pca = pca.fit_transform(embeddings_palabras)

# Visualizar
fig, ax = plt.subplots(figsize=(14, 10))

scatter = ax.scatter(embeddings_2d_pca[:, 0], embeddings_2d_pca[:, 1],
                     c=range(len(top50_palabras)), cmap='viridis',
                     s=100, alpha=0.6, edgecolors='black', linewidth=0.5)

# Añadir etiquetas solo para Top 20
for i in range(min(20, len(top50_palabras))):
    ax.annotate(top50_palabras[i],
                (embeddings_2d_pca[i, 0], embeddings_2d_pca[i, 1]),
                fontsize=9, alpha=0.8)

ax.set_xlabel('Componente Principal 1', fontsize=12)
ax.set_ylabel('Componente Principal 2', fontsize=12)
ax.set_title('Visualización de Embeddings - Top 50 Palabras (PCA)',
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.colorbar(scatter, ax=ax, label='Ranking de Frecuencia')
plt.tight_layout()
plt.savefig('/content/embeddings_visualizacion.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualización guardada: embeddings_visualizacion.png\n")

# ============================================================================
# CELDA 12: PASO 5 - TOPIC MODELING (LDA)
# ============================================================================

print("\n" + "="*70)
print("🎯 PASO 5: TOPIC MODELING (Latent Dirichlet Allocation)")
print("="*70 + "\n")

# Preparar textos para LDA (usar texto sin stopwords)
textos_lda = df['palabras_sin_stopwords'].tolist()

print(f"📚 Documentos para análisis: {len(textos_lda)}\n")

# Crear matriz de términos (Count Vectorizer)
print("⏳ Creando matriz de términos...\n")

vectorizer = CountVectorizer(max_features=100,
                             min_df=2,  # Mínimo 2 documentos
                             max_df=0.8)  # Máximo 80% de documentos

matriz_terminos = vectorizer.fit_transform(textos_lda)
vocabulario = vectorizer.get_feature_names_out()

print(f"✅ Matriz creada: {matriz_terminos.shape}")
print(f"   - {matriz_terminos.shape[0]} documentos")
print(f"   - {matriz_terminos.shape[1]} términos únicos\n")

# Aplicar LDA
print("⏳ Aplicando Topic Modeling (LDA)...\n")

n_topics = 5  # Número de tópicos a identificar

lda_model = LatentDirichletAllocation(n_components=n_topics,
                                      random_state=42,
                                      max_iter=20)

lda_output = lda_model.fit_transform(matriz_terminos)

print(f"✅ Modelo LDA entrenado con {n_topics} tópicos\n")

⏳ Reduciendo dimensionalidad con PCA...

✅ Visualización guardada: embeddings_visualizacion.png


🎯 PASO 5: TOPIC MODELING (Latent Dirichlet Allocation)



KeyError: 'palabras_sin_stopwords'

In [ ]:
# Función para obtener top palabras por tópico
def obtener_top_palabras(model, feature_names, n_top_words=10):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)
    return topics

# Obtener top palabras por tópico
top_palabras_por_topico = obtener_top_palabras(lda_model, vocabulario, n_top_words=10)

# Mostrar tópicos
for idx, palabras in enumerate(top_palabras_por_topico, 1):
    print(f"🎯 TÓPICO {idx}:")
    print(f"   Palabras clave: {', '.join(palabras)}\n")

# Visualización de tópicos
fig, axes = plt.subplots(n_topics, 1, figsize=(12, 3*n_topics))

if n_topics == 1:
    axes = [axes]

for idx, (topic, palabras) in enumerate(zip(lda_model.components_, top_palabras_por_topico)):
    top_indices = topic.argsort()[-10:][::-1]
    top_values = topic[top_indices]

    axes[idx].barh(range(10), top_values[::-1], color=plt.cm.viridis(idx/n_topics))
    axes[idx].set_yticks(range(10))
    axes[idx].set_yticklabels([vocabulario[i] for i in top_indices][::-1])
    axes[idx].set_xlabel('Peso en el tópico')
    axes[idx].set_title(f'Tópico {idx+1}', fontweight='bold')
    axes[idx].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/topic_modeling_resultados.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Obtener tópico dominante por documento
topico_dominante = np.argmax(lda_output, axis=1)
probabilidad_dominante = np.max(lda_output, axis=1)

# Añadir al DataFrame
df['topico_dominante'] = topico_dominante + 1  # +1 para indexar desde 1
df['probabilidad_topico'] = probabilidad_dominante

print("✅ Tópicos asignados a documentos\n")

# Distribución de documentos por tópico
print("📊 DISTRIBUCIÓN DE DOCUMENTOS POR TÓPICO:\n")
distribucion_topicos = df['topico_dominante'].value_counts().sort_index()

for topico, cantidad in distribucion_topicos.items():
    porcentaje = (cantidad / len(df)) * 100
    print(f"Tópico {topico}: {cantidad} documentos ({porcentaje:.1f}%)")

# Visualización
plt.figure(figsize=(10, 6))
distribucion_topicos.plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Tópico', fontsize=12)
plt.ylabel('Número de Documentos', fontsize=12)
plt.title('Distribución de Documentos por Tópico', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/distribucion_topicos.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualización guardada: distribucion_topicos.png\n")

# Mostrar ejemplos por tópico
print("\n🔍 EJEMPLOS DE DOCUMENTOS POR TÓPICO (primeros 200 caracteres):\n")
for topico in range(1, n_topics + 1):
    docs_topico = df[df['topico_dominante'] == topico]
    if len(docs_topico) > 0:
        print(f"--- TÓPICO {topico} ---")
        ejemplo = docs_topico.iloc[0]['texto_completo_lematizado'][:200]
        print(f"{ejemplo}...\n")


In [ ]:
# Crear DataFrame con resumen de resultados
resultados_analisis = {
    'top20_palabras_sin_stopwords': df_top20_sin_stop,
    'top20_palabras_con_stopwords': df_top20_con_stop,
    'top20_oraciones': df_top20_oraciones,
    'topicos_identificados': pd.DataFrame({
        'Topico': range(1, n_topics + 1),
        'Palabras_clave': [', '.join(palabras[:5]) for palabras in top_palabras_por_topico]
    }),
    'distribucion_documentos_topicos': distribucion_topicos.to_frame('Cantidad')
}

# Guardar en Excel con múltiples hojas
ARCHIVO_RESULTADOS = "/content/resultados_analisis_completo.xlsx"

with pd.ExcelWriter(ARCHIVO_RESULTADOS, engine='openpyxl') as writer:
    # Datos originales con tópicos
    df.to_excel(writer, sheet_name='Datos_con_topicos', index=False)

    # Resultados de análisis
    resultados_analisis['top20_palabras_sin_stopwords'].to_excel(
        writer, sheet_name='Top20_sin_stopwords', index=False)

    resultados_analisis['top20_palabras_con_stopwords'].to_excel(
        writer, sheet_name='Top20_con_stopwords', index=False)

    resultados_analisis['top20_oraciones'].to_excel(
        writer, sheet_name='Top20_oraciones', index=False)

    resultados_analisis['topicos_identificados'].to_excel(
        writer, sheet_name='Topicos', index=False)

    resultados_analisis['distribucion_documentos_topicos'].to_excel(
        writer, sheet_name='Dist_topicos')

print(f"✅ Resultados guardados en: {ARCHIVO_RESULTADOS}\n")

print("="*70)
print("✅ ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*70 + "\n")

print("📊 RESUMEN DEL ANÁLISIS:")
print(f"  • Total de palabras (sin stopwords): {len(todas_palabras_sin_stop):,}")
print(f"  • Vocabulario único: {len(frecuencias_sin_stop):,}")
print(f"  • Total de oraciones: {len(todas_oraciones):,}")
print(f"  • Tópicos identificados: {n_topics}")
print(f"  • Documentos analizados: {len(df)}\n")

print("📁 ARCHIVOS GENERADOS:")
print("  • top20_palabras_comparacion.png")
print("  • nube_palabras.png")
print("  • embeddings_visualizacion.png")
print("  • topic_modeling_resultados.png")
print("  • distribucion_topicos.png")
print(f"  • {ARCHIVO_RESULTADOS}\n")

PARTE 2: análisis semántico

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.util import bigrams
import re

# Descargar stopwords en español
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('spanish'))

# Cargar el archivo Excel
file_path = '/content/Corpus_humano.xlsx'
df = pd.read_excel(file_path)

print("Archivo cargado exitosamente")
print(f"Dimensiones: {df.shape}")
print(f"\nColumnas disponibles:\n{df.columns.tolist()}")


In [ ]:
def limpiar_texto(texto):
    """Limpia y normaliza el texto manteniendo acentos españoles"""
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    # Mantener letras españolas (incluyendo ñ y vocales acentuadas)
    texto = re.sub(r'[^a-záéíóúñü\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def extraer_bigramas_con_stopwords(texto):
    """Extrae bigramas que contengan al menos una stopword"""
    texto_limpio = limpiar_texto(texto)
    if not texto_limpio:
        return []

    palabras = texto_limpio.split()
    if len(palabras) < 2:
        return []

    bigramas_encontrados = []
    for bigrama in bigrams(palabras):
        # Verificar si al menos una palabra del bigrama es stopword
        if bigrama[0] in stop_words or bigrama[1] in stop_words:
            bigramas_encontrados.append(f"{bigrama[0]}_{bigrama[1]}")

    return bigramas_encontrados

def contar_bigramas_stopwords(texto):
    """Cuenta el número de bigramas con stopwords"""
    return len(extraer_bigramas_con_stopwords(texto))

# Procesar cada columna de texto
columnas_texto = df.select_dtypes(include=['object']).columns

print(f"\n{'='*60}")
print("PROCESANDO COLUMNAS: Resumen, Introducción, Conclusiones")
print(f"{'='*60}\n")

In [ ]:
# Verificar que las columnas existen
columnas_esperadas = ['Resumen', 'Introducción', 'Conclusiones']
columnas_disponibles = [col for col in columnas_esperadas if col in df.columns]
print(f"Columnas encontradas: {columnas_disponibles}\n")

for columna in columnas_texto:
    print(f"Procesando columna: '{columna}'")

    # Crear nuevas columnas para esta columna original
    col_bigramas = f"{columna}_bigramas_stopwords"
    col_count = f"{columna}_count_bigramas"
    col_lista = f"{columna}_lista_bigramas"

    # Extraer bigramas
    df[col_lista] = df[columna].apply(extraer_bigramas_con_stopwords)

    # Contar bigramas
    df[col_count] = df[col_lista].apply(len)

    # Unir bigramas en string
    df[col_bigramas] = df[col_lista].apply(lambda x: ', '.join(x) if x else '')

    # Mostrar estadísticas
    total_bigramas = df[col_count].sum()
    promedio = df[col_count].mean()
    print(f"  ✓ Total bigramas encontrados: {total_bigramas}")
    print(f"  ✓ Promedio por fila: {promedio:.2f}\n")

In [ ]:
# Mostrar muestra de resultados
print(f"\n{'='*60}")
print("MUESTRA DE RESULTADOS (primeras 3 filas)")
print(f"{'='*60}\n")

nuevas_columnas = [col for col in df.columns if 'bigramas' in col.lower()]
print(df[nuevas_columnas].head(3))

# Guardar resultado
output_path = '/content/Corpus_humano_con_bigramas.xlsx'
df.to_excel(output_path, index=False)
print(f"\n{'='*60}")
print(f"✓ Archivo guardado exitosamente en:")
print(f"  {output_path}")
print(f"{'='*60}")

In [ ]:
# Resumen final
print(f"\n{'='*60}")
print("RESUMEN FINAL")
print(f"{'='*60}")
print(f"Total de filas procesadas: {len(df)}")
print(f"Columnas procesadas: {', '.join(columnas_disponibles)}")
print(f"Nuevas columnas creadas: {len(nuevas_columnas)}")
print(f"\nEstructura de columnas nuevas:")
print(f"  Por cada columna original (Resumen, Introducción, Conclusiones):")
print(f"    - [columna]_bigramas_stopwords: texto con bigramas")
print(f"    - [columna]_count_bigramas: cantidad de bigramas")
print(f"    - [columna]_lista_bigramas: lista de bigramas")
print(f"\nTotal nuevas columnas: {len(nuevas_columnas)}")
for i, col in enumerate(nuevas_columnas, 1):
    print(f"  {i}. {col}")

Análisis de coherencia resumen introducción versus conclusiones





In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

In [ ]:
print("="*70)
print("ANÁLISIS DE COHERENCIA - TEXTO ORIGINAL COMPLETO")
print("Resumen + Introducción vs Conclusiones")
print("="*70)
print(f"\nArchivo cargado: {len(df)} filas")
print(f"Columnas: {df.columns.tolist()}\n")

In [ ]:
def limpiar_texto(texto):
    """Limpia y normaliza el texto en español"""
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = re.sub(r'[^a-záéíóúñü\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def calcular_coherencia(resumen, introduccion, conclusiones):
    """
    Calcula coherencia entre (Resumen+Introducción) vs Conclusiones
    usando similitud de coseno con TF-IDF sobre el TEXTO COMPLETO ORIGINAL

    Retorna: valor entre 0 y 1 (0=sin coherencia, 1=coherencia perfecta)
    """
    # Usar texto ORIGINAL completo (solo convertir a string)
    texto_inicio = str(resumen) + " " + str(introduccion)
    texto_conclusion = str(conclusiones)

    # Limpiar solo para eliminar NaN
    if texto_inicio == "nan nan" or texto_conclusion == "nan":
        return 0.0

    # Si los textos son muy cortos, retornar coherencia baja
    if len(texto_inicio.split()) < 5 or len(texto_conclusion.split()) < 5:
        return 0.0

    try:
        # Crear vectorizador TF-IDF con texto COMPLETO en español
        vectorizer = TfidfVectorizer(
            max_features=500,  # Más features para capturar texto completo
            ngram_range=(1, 3),  # unigramas, bigramas y trigramas
            min_df=1,
            lowercase=True,
            strip_accents=None  # Mantener acentos españoles
        )

        # Vectorizar ambos textos COMPLETOS
        vectors = vectorizer.fit_transform([texto_inicio, texto_conclusion])

        # Calcular similitud de coseno
        similitud = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]

        return round(similitud, 4)

    except:
        return 0.0

In [ ]:
# Calcular coherencia para cada fila
print("Procesando coherencia por fila...\n")

coherencias = []
for idx, row in df.iterrows():
    coherencia = calcular_coherencia(
        row.get('Resumen', ''),
        row.get('Introducción', ''),
        row.get('Conclusiones', '')
    )
    coherencias.append(coherencia)

    if idx < 3:  # Mostrar primeros 3 ejemplos
        print(f"Fila {idx + 1}: Coherencia = {coherencia:.4f}")

In [ ]:
# Agregar nueva columna
df['Coherencia_Resumen_Intro_vs_Conclusiones'] = coherencias

# Interpretación de la coherencia
def interpretar_coherencia(valor):
    """Interpreta el valor de coherencia"""
    if valor >= 0.7:
        return "Alta"
    elif valor >= 0.4:
        return "Media"
    elif valor > 0:
        return "Baja"
    else:
        return "Sin datos"

df['Interpretacion_Coherencia'] = df['Coherencia_Resumen_Intro_vs_Conclusiones'].apply(interpretar_coherencia)

In [ ]:
# Estadísticas
print(f"\n{'='*70}")
print("ESTADÍSTICAS DE COHERENCIA")
print("="*70)
print(f"Coherencia promedio: {np.mean(coherencias):.4f}")
print(f"Coherencia mínima: {np.min(coherencias):.4f}")
print(f"Coherencia máxima: {np.max(coherencias):.4f}")
print(f"Desviación estándar: {np.std(coherencias):.4f}")

# Distribución
print(f"\n{'='*70}")
print("DISTRIBUCIÓN DE COHERENCIA")
print("="*70)
interpretaciones = df['Interpretacion_Coherencia'].value_counts()
for nivel, cantidad in interpretaciones.items():
    porcentaje = (cantidad / len(df)) * 100
    print(f"{nivel:15s}: {cantidad:3d} documentos ({porcentaje:.1f}%)")


In [ ]:
# Mostrar muestra de resultados
print(f"\n{'='*70}")
print("MUESTRA DE RESULTADOS (primeras 5 filas)")
print("="*70)
columnas_mostrar = ['Coherencia_Resumen_Intro_vs_Conclusiones', 'Interpretacion_Coherencia']
if all(col in df.columns for col in columnas_mostrar):
    print(df[columnas_mostrar].head(5))

# Guardar resultado
output_path = '/content/Corpus_humano_con_coherencia.xlsx'
df.to_excel(output_path, index=False)

print(f"\n{'='*70}")
print("✓ ARCHIVO GUARDADO EXITOSAMENTE")
print("="*70)
print(f"Ubicación: {output_path}")
print(f"Total de filas: {len(df)}")
print(f"Nuevas columnas agregadas:")
print(f"  1. Coherencia_Resumen_Intro_vs_Conclusiones (valor 0-1)")
print(f"  2. Interpretacion_Coherencia (Alta/Media/Baja)")
print("="*70)

# Interpretación de la métrica
print(f"\n{'='*70}")
print("INTERPRETACIÓN DE LA MÉTRICA")
print("="*70)
print("Valores de Coherencia:")
print("  0.70 - 1.00 → Alta coherencia (contenido muy relacionado)")
print("  0.40 - 0.69 → Media coherencia (contenido parcialmente relacionado)")
print("  0.01 - 0.39 → Baja coherencia (contenido poco relacionado)")
print("  0.00        → Sin datos suficientes")
print("="*70)